# "THE SALARY IS RIGHT" — Full Pipeline

Predict annual salary from job posting descriptions using progressively better models.

**Dataset:** `lukebarousse/data_jobs` — 786K real-world job postings from 2023

**Pipeline:**
1. Data Curation — load, filter, deduplicate, split
2. Data Preprocessing — rewrite descriptions with an LLM
3. Baselines & Traditional ML — random, constant, linear regression, bag of words, Random Forest, XGBoost
4. Neural Network — vanilla 8-layer NN trained with PyTorch
5. Frontier LLMs — GPT-4.1-nano, Claude, GPT-4.1, Gemini out of the box
6. Fine-tuning — supervised fine-tuning of GPT-4.1-nano via OpenAI API

**Evaluation metric:** Average absolute error in dollars (predicted salary vs actual salary)

In [ ]:
# All imports
import os
import random
import json
import csv
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from collections import Counter
from tqdm.notebook import tqdm
from datasets import load_dataset
from dotenv import load_dotenv
from litellm import completion
from openai import OpenAI
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from salary_predictor.jobs import Job
from salary_predictor.parser import parse
from salary_predictor.evaluator import evaluate

load_dotenv(override=True)

---
# Part 1: Data Curation

In [ ]:
dataset = load_dataset('lukebarousse/data_jobs', split='train')
print(f"Total job postings: {len(dataset):,}")
print(f"Columns: {dataset.column_names}")

In [ ]:
# Quick stats on salary availability
df = dataset.to_pandas()
print(f"Rows with yearly salary: {df['salary_year_avg'].notna().sum():,} / {len(df):,}")
print(f"\nSalary stats (for rows that have it):")
print(df['salary_year_avg'].describe())
print(f"\nJob categories:")
print(df['job_title_short'].value_counts())

In [ ]:
# Investigate a sample row with salary
sample = df[df['salary_year_avg'].notna()].iloc[0]
for col in df.columns:
    val = sample[col]
    if isinstance(val, str) and len(val) > 300:
        val = val[:300] + '...'
    print(f"  {col}: {val}")

In [ ]:
# Parse all rows into Job objects
items = []
for i in tqdm(range(len(dataset))):
    job = parse(dataset[i])
    if job is not None:
        items.append(job)

print(f"Parsed {len(items):,} valid jobs from {len(dataset):,} postings")

In [ ]:
print(items[0])
print()
print(items[0].full)

In [ ]:
# Salary distribution
salaries = [job.salary for job in items]

plt.figure(figsize=(15, 6))
plt.title(f"Salaries: Avg ${sum(salaries)/len(salaries):,.0f} | Min ${min(salaries):,.0f} | Max ${max(salaries):,.0f}")
plt.xlabel('Annual Salary ($)')
plt.ylabel('Count')
plt.hist(salaries, rwidth=0.7, color='orange', bins=range(0, 510_000, 10_000))
plt.show()

In [ ]:
# Category distribution
category_counts = Counter([job.category for job in items])
categories = list(category_counts.keys())
counts = [category_counts[c] for c in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color='goldenrod')
plt.title('Jobs per Category')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.show()

In [ ]:
# Remove duplicates
print(f"Before dedup: {len(items):,}")
seen = set()
unique_items = []
for item in items:
    key = (item.title, item.company, item.salary)
    if key not in seen:
        seen.add(key)
        unique_items.append(item)
items = unique_items
print(f"After dedup: {len(items):,}")

# Shuffle and split
random.seed(42)
random.shuffle(items)

n = len(items)
val_size = min(2000, int(n * 0.05))
test_size = min(2000, int(n * 0.05))
train_size = n - val_size - test_size

train = items[:train_size]
val = items[train_size:train_size + val_size]
test = items[train_size + val_size:]

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

---
# Part 2: Data Preprocessing with LLM

Rewrite raw job posting text into a clean structured format using an LLM.

**This step is optional** — if you want to skip it (to save cost/time), the models below will just use the original `full` text. Set `SKIP_PREPROCESSING = True`.

In [ ]:
SKIP_PREPROCESSING = True  # Set to False to run LLM preprocessing

SYSTEM_PROMPT = """Create a concise description of a job posting. Respond only in this format:
Title: Standardized job title
Category: e.g. Data Science, Software Engineering, Management
Level: e.g. Junior, Mid, Senior, Lead, Director
Company: Company name
Location: City, State or Remote
Skills: Key technical skills required
Summary: 1 sentence description of the role"""

In [ ]:
if not SKIP_PREPROCESSING:
    # Test on one item first
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
    response = completion(messages=messages, model="openai/gpt-4.1-nano")
    print("=== ORIGINAL ===")
    print(items[0].full)
    print("\n=== PREPROCESSED ===")
    print(response.choices[0].message.content)

In [ ]:
all_items = train + val + test

if not SKIP_PREPROCESSING:
    MODEL = "openai/gpt-4.1-nano"
    for item in tqdm(all_items):
        if not item.summary:
            try:
                msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}]
                resp = completion(messages=msgs, model=MODEL)
                item.summary = resp.choices[0].message.content
            except Exception as e:
                item.summary = item.full
else:
    # Use original text as summary
    for item in all_items:
        if not item.summary:
            item.summary = item.full

print(f"All {len(all_items):,} items have summaries")

---
# Part 3: Baselines & Traditional ML

Get your piece of paper ready to write down each result!

In [ ]:
def get_text(job):
    """Helper to get best available text for a job"""
    return job.summary or job.full or ''

### Model 1: Random Predictor

In [ ]:
def random_predictor(job):
    return random.randrange(15_000, 500_000)

random.seed(42)
evaluate(random_predictor, test)

### Model 2: Constant Predictor (guess the average)

In [ ]:
training_average = sum(j.salary for j in train) / len(train)
print(f"Training average salary: ${training_average:,.0f}")

def constant_predictor(job):
    return training_average

evaluate(constant_predictor, test)

### Model 3: Linear Regression (hand-crafted features)

In [ ]:
def get_features(job):
    title_lower = (job.title or '').lower()
    return {
        'text_length': len(get_text(job)),
        'is_remote': 1 if job.work_from_home else 0,
        'is_senior': 1 if 'senior' in title_lower or 'sr.' in title_lower else 0,
        'is_lead': 1 if any(w in title_lower for w in ['lead', 'principal', 'staff']) else 0,
        'is_director': 1 if any(w in title_lower for w in ['director', 'vp', 'head']) else 0,
        'is_manager': 1 if 'manager' in title_lower else 0,
        'is_junior': 1 if any(w in title_lower for w in ['junior', 'jr.', 'entry', 'intern']) else 0,
        'num_skills': len((job.skills or '').split(',')) if job.skills else 0,
    }

train_df = pd.DataFrame([get_features(j) for j in train])
train_df['salary'] = [j.salary for j in train]

feature_cols = [c for c in train_df.columns if c != 'salary']
lr_model = LinearRegression()
lr_model.fit(train_df[feature_cols], train_df['salary'])

for feat, coef in zip(feature_cols, lr_model.coef_):
    print(f"  {feat}: {coef:,.0f}")
print(f"  Intercept: {lr_model.intercept_:,.0f}")

def linear_regression_predictor(job):
    feats = pd.DataFrame([get_features(job)])
    return max(0, lr_model.predict(feats)[0])

evaluate(linear_regression_predictor, test)

### Model 4: NLP + Linear Regression (Bag of Words)

In [ ]:
prices = np.array([float(j.salary) for j in train])
documents = [get_text(j) for j in train]

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

words = vectorizer.get_feature_names_out()
print(f"Vocabulary: {len(words)} words. Sample: {list(words[500:520])}")

nlp_regressor = LinearRegression()
nlp_regressor.fit(X, prices)

def nlp_linear_regression_predictor(job):
    x = vectorizer.transform([get_text(job)])
    return max(0, nlp_regressor.predict(x)[0])

evaluate(nlp_linear_regression_predictor, test)

### Model 5: Random Forest

In [ ]:
subset = min(15_000, len(train))
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

def random_forest_predictor(job):
    x = vectorizer.transform([get_text(job)])
    return max(0, rf_model.predict(x)[0])

evaluate(random_forest_predictor, test)

### Model 6: XGBoost

In [ ]:
np.random.seed(42)
xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

def xgboost_predictor(job):
    x = vectorizer.transform([get_text(job)])
    return max(0, xgb_model.predict(x)[0])

evaluate(xgboost_predictor, test)

---
# Part 4: Vanilla Neural Network

In [ ]:
y_nn = np.array([float(j.salary) for j in train])
docs_nn = [get_text(j) for j in train]

np.random.seed(42)
hash_vectorizer = HashingVectorizer(n_features=2000, stop_words='english', binary=True)
X_nn = hash_vectorizer.fit_transform(docs_nn)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.layers(x)

X_tensor = torch.FloatTensor(X_nn.toarray())
y_tensor = torch.FloatTensor(y_nn).unsqueeze(1)
X_tr, X_vl, y_tr, y_vl = train_test_split(X_tensor, y_tensor, test_size=0.01, random_state=42)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
nn_model = NeuralNetwork(X_tensor.shape[1])
print(f"Parameters: {sum(p.numel() for p in nn_model.parameters() if p.requires_grad):,}")

In [ ]:
loss_fn = nn.MSELoss()
optimizer = optim.Adam(nn_model.parameters(), lr=0.001)
EPOCHS = 2

for epoch in range(EPOCHS):
    nn_model.train()
    for bx, by in tqdm(train_loader):
        optimizer.zero_grad()
        out = nn_model(bx)        # forward pass
        loss = loss_fn(out, by)    # loss calculation
        loss.backward()            # backward pass
        optimizer.step()           # optimization

    nn_model.eval()
    with torch.no_grad():
        vl_loss = loss_fn(nn_model(X_vl), y_vl)
    print(f'Epoch {epoch+1}/{EPOCHS} — Train Loss: {loss.item():.0f}, Val Loss: {vl_loss.item():.0f}')

In [ ]:
def neural_network_predictor(job):
    nn_model.eval()
    with torch.no_grad():
        vec = hash_vectorizer.transform([get_text(job)])
        vec = torch.FloatTensor(vec.toarray())
        return max(0, nn_model(vec)[0].item())

evaluate(neural_network_predictor, test)

---
# Part 5: Frontier LLMs (Out of the Box)

No training — just asking frontier models to estimate salary using their world knowledge.

**Requires API keys** in your `.env` file. Adjust model names to whatever is current for you.

In [ ]:
def messages_for(job):
    text = get_text(job)
    msg = f"Estimate the annual salary for this job posting in USD. Respond with only the number, no explanation\n\n{text}"
    return [{"role": "user", "content": msg}]

In [ ]:
def gpt_4__1_nano(job):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(job))
    return response.choices[0].message.content

evaluate(gpt_4__1_nano, test)

In [ ]:
def claude_sonnet(job):
    response = completion(model="anthropic/claude-sonnet-4-20250514", messages=messages_for(job))
    return response.choices[0].message.content

evaluate(claude_sonnet, test, size=50, workers=2)

In [ ]:
def gpt_4__1(job):
    response = completion(model="openai/gpt-4.1", messages=messages_for(job))
    return response.choices[0].message.content

evaluate(gpt_4__1, test)

In [ ]:
def gemini_2__5_flash(job):
    response = completion(model="gemini/gemini-2.5-flash", messages=messages_for(job))
    return response.choices[0].message.content

evaluate(gemini_2__5_flash, test)

---
# Part 6: Fine-tuning GPT-4.1-nano

Supervised fine-tuning via OpenAI API. Uses 100 examples (costs pennies).

In [ ]:
openai_client = OpenAI()

fine_tune_train = train[:100]
fine_tune_val = val[:50]

In [ ]:
def ft_messages_for(job):
    text = get_text(job)
    msg = f"Estimate the annual salary for this job posting in USD. Respond with only the salary, no explanation\n\n{text}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${job.salary:,.0f}"}
    ]

def write_jsonl(jobs, filename):
    os.makedirs('jsonl', exist_ok=True)
    with open(filename, 'w') as f:
        for job in jobs:
            f.write(json.dumps({"messages": ft_messages_for(job)}) + '\n')

write_jsonl(fine_tune_train, 'jsonl/ft_train.jsonl')
write_jsonl(fine_tune_val, 'jsonl/ft_val.jsonl')
print("JSONL files written")

In [ ]:
with open('jsonl/ft_train.jsonl', 'rb') as f:
    train_file = openai_client.files.create(file=f, purpose='fine-tune')
with open('jsonl/ft_val.jsonl', 'rb') as f:
    val_file = openai_client.files.create(file=f, purpose='fine-tune')

print(f"Train file: {train_file.id} | Val file: {val_file.id}")

In [ ]:
ft_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="salary"
)
print(f"Fine-tuning job: {ft_job.id}")

In [ ]:
# Run this cell repeatedly until status is 'succeeded'
job_id = openai_client.fine_tuning.jobs.list(limit=1).data[0].id
status = openai_client.fine_tuning.jobs.retrieve(job_id)
print(f"Status: {status.status}")
print(f"Fine-tuned model: {status.fine_tuned_model}")

In [ ]:
fine_tuned_model_name = openai_client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

def gpt_4__1_nano_fine_tuned(job):
    text = get_text(job)
    msg = f"Estimate the annual salary for this job posting in USD. Respond with only the salary, no explanation\n\n{text}"
    response = openai_client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=[{"role": "user", "content": msg}],
        max_tokens=10
    )
    return response.choices[0].message.content

# Quick sanity check
print(f"Actual: ${test[0].salary:,.0f}")
print(f"Predicted: {gpt_4__1_nano_fine_tuned(test[0])}")

In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

---
# Final Results Comparison

Fill in your actual error numbers from each model above!

In [ ]:
# Replace zeros with your actual results
results = [
    ("Constant", "gray", 0),
    ("Linear Reg", "gray", 0),
    ("NLP + LR", "gray", 0),
    ("Random Forest", "gray", 0),
    ("XGBoost", "gray", 0),
    ("Neural Net", "orange", 0),
    ("GPT 4.1 Nano", "slateblue", 0),
    ("Claude Sonnet", "slateblue", 0),
    ("GPT 4.1", "green", 0),
    ("Gemini Flash", "slateblue", 0),
    ("GPT 4.1 Nano (FT)", "red", 0),
]

labels, colors, values = zip(*results)
fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
fig.update_layout(
    title="The Salary Is Right — Model Comparison",
    yaxis=dict(range=[0, max(values) * 1.1 if max(values) > 0 else 100_000], title="Average Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1000, height=600
)
fig.show()